<a href="https://colab.research.google.com/github/vasanthiginkala5/ADVANCED-GEN-AI-NLP_ASSIGNMENT/blob/main/Fine_tune_a_transformer_model_(BERT_DistilBERT)_to_perform_Part_of_Speech_(POS)_Tagging_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:







!pip install transformers datasets seqeval accelerate -q

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import classification_report

In [ ]:
dataset = load_dataset("wikiann", "en")
dataset

README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

In [ ]:
label_list = dataset["train"].features["ner_tags"].feature.names

num_labels = len(label_list)

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)

Device: cuda


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)

        previous = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous = word_idx

        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
train_dataset = tokenized_dataset["train"].select(range(2000))
eval_dataset = tokenized_dataset["validation"].select(range(500))

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    fp16=True
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.381011
2,0.545165,0.334395


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.5451646728515624, metrics={'train_runtime': 55.9022, 'train_samples_per_second': 71.554, 'train_steps_per_second': 8.944, 'total_flos': 130664911872000.0, 'train_loss': 0.5451646728515624, 'epoch': 2.0})

In [ ]:
preds, labels, _ = trainer.predict(eval_dataset)

preds = np.argmax(preds, axis=2)

true_labels = [
    [label_list[l] for l in label if l != -100]
    for label in labels
]

true_preds = [
    [label_list[p] for p, l in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]

print(classification_report(true_labels, true_preds))

              precision    recall  f1-score   support

         LOC       0.69      0.75      0.72       203
         ORG       0.56      0.64      0.60       225
         PER       0.83      0.83      0.83       242

   micro avg       0.69      0.74      0.72       670
   macro avg       0.70      0.74      0.72       670
weighted avg       0.70      0.74      0.72       670



In [ ]:
sentence = "John works at Google in California"
tokens = sentence.split()

inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
inputs = {k:v.to(device) for k,v in inputs.items()}

outputs = model(**inputs).logits
preds = torch.argmax(outputs, dim=2)

for token, pred in zip(tokens, preds[0]):
    print(token, "->", id2label[pred.item()])

John -> O
works -> B-PER
at -> I-PER
Google -> O
in -> B-ORG
California -> I-ORG


In [ ]:
model.save_pretrained("nlp_model")
tokenizer.save_pretrained("nlp_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('nlp_model/tokenizer_config.json', 'nlp_model/tokenizer.json')

POS Tagging vs Chunking

POS Tagging:

Word-level grammatical labeling
Example: NOUN, VERB, ADJ

Chunking:

Groups words into phrases
Example: NP (noun phrase), VP (verb phrase)

POS Tagging:
Level:Word,Output:
Grammar tag,Complexity:
Easy

Chunking:Level:Phrase,Output:
Phrase group,Complexity
Medium

⚠️ Challenges Faced (Short)
Dataset loading issues due to Hugging Face and Python 3.12 compatibility.
Difficulty in aligning subword tokens with correct labels using BERT tokenizer.
High memory usage causing RAM issues during training.
Need to properly configure GPU for faster training.
Token-level prediction was more complex than normal classification tasks.

📊 Observations and Insights (Short)
DistilBERT performs well for POS tagging and token classification tasks.
Subword tokenization improves understanding but requires careful label alignment.
POS tagging is easier compared to chunking, which involves phrase-level grouping.
Even small training runs give good results due to pre-trained transformer models.
Proper preprocessing is the key factor for good model performance.

🔄 Expected Pipeline Flow

Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

Raw data is first tokenized using BERT tokenizer. Then labels are aligned with subword tokens. After preprocessing, the model is trained using transformer architecture. The trained model is evaluated using metrics like precision, recall, and F1-score. Finally, inference is performed on custom sentences and results are compared between POS tagging and chunking tasks.